# Object Sorting and Neural Network Layer Abstraction

**Session:** 2026-10-30 — OOP with Python  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

This notebook brings together object-oriented principles across two practical dimensions:
1. **Custom Object Sorting:** Sorting domain objects in Python collections using key functions, operator getters, and rich comparison dunder methods (`__lt__`, `__eq__`).
2. **Applied AI Case Study:** Building a modular neural network layer hierarchy from scratch using pure Python and NumPy, mirroring the architectural design of modern deep learning frameworks (such as PyTorch's `nn.Module`).

## Learning Outcomes

- Sort collections of custom domain objects using lambda expressions and `operator.attrgetter`.
- Implement rich comparison dunders (`__lt__`, `__eq__`) with `@functools.total_ordering`.
- Parse and sort structured book records from `data/books_catalog.json`.
- Design an OOP neural network framework with a base `Module`, a `Linear` dense layer, a `ReLU` activation, and a composite `Sequential` pipeline.
- Verify end-to-end forward pass execution, parameter registration, and tensor shape transformations.

## 1. Custom Object Sorting in Python Collections

When storing custom class objects in Python lists, invoking `list.sort()` or `sorted()` requires defining how objects should be compared. Python provides three clean patterns:

In [ ]:
import operator
from functools import total_ordering
from typing import Any


class Square:
    """Geometric square with side length and area calculation."""

    def __init__(self, side: float) -> None:
        self.side = float(side)

    @property
    def area(self) -> float:
        return self.side * self.side

    def __repr__(self) -> str:
        return f"Square(side={self.side}, area={self.area})"


squares = [Square(5), Square(1), Square(8), Square(3)]

# Approach A: Sorting with a lambda key
by_side = sorted(squares, key=lambda s: s.side)
print("Sorted by side (lambda):", by_side)
assert [s.side for s in by_side] == [1.0, 3.0, 5.0, 8.0]

# Approach B: Sorting with operator.attrgetter
by_area_desc = sorted(squares, key=operator.attrgetter("area"), reverse=True)
print("Sorted by area desc (attrgetter):", by_area_desc)
assert [s.side for s in by_area_desc] == [8.0, 5.0, 3.0, 1.0]

### Approach C: Natural Ordering via `@total_ordering` and `__lt__`

Implementing `__lt__` and `__eq__` on the class, decorated with `@functools.total_ordering`, gives custom objects a natural sort order and enables standard comparison operators (`<`, `<=`, `>`, `>=`, `==`):

In [ ]:
import json
from pathlib import Path


@total_ordering
class Book:
    """Book entity with natural ordering by rating (descending), then price (ascending)."""

    def __init__(
        self,
        isbn: str,
        title: str,
        author: str,
        genre: str,
        price: float,
        rating: float,
        publication_year: int,
    ) -> None:
        self.isbn = isbn
        self.title = title
        self.author = author
        self.genre = genre
        self.price = float(price)
        self.rating = float(rating)
        self.publication_year = int(publication_year)

    def __eq__(self, other: Any) -> bool:
        if not isinstance(other, Book):
            return NotImplemented
        return (self.rating, -self.price) == (other.rating, -other.price)

    def __lt__(self, other: Any) -> bool:
        if not isinstance(other, Book):
            return NotImplemented
        # Primary sort: higher rating first; Secondary: lower price first
        return (-self.rating, self.price) < (-other.rating, other.price)

    def __repr__(self) -> str:
        return f"Book('{self.title[:25]}...', rating={self.rating}, price=${self.price:.2f})"


# Load books from dataset
candidates = [
    Path("../data/books_catalog.json"),
    Path("data/books_catalog.json"),
    Path("python-to-ai/01-python-fundamentals/2026-10-30-oop-with-python/data/books_catalog.json"),
    Path("01-python-fundamentals/2026-10-30-oop-with-python/data/books_catalog.json"),
]
books_file = next((p for p in candidates if p.exists()), Path("data/books_catalog.json"))
with books_file.open("r", encoding="utf-8") as f:
    raw_books = json.load(f)

catalog = [
    Book(
        isbn=b["isbn"],
        title=b["title"],
        author=b["author"],
        genre=b["genre"],
        price=b["price"],
        rating=b["rating"],
        publication_year=b["publication_year"],
    )
    for b in raw_books
]

# Natural sort using Book's dunder comparisons
catalog.sort()

print("Top 3 books by rating & price:")
for b in catalog[:3]:
    print(" ", b)

# Verify top rated
assert catalog[0].rating >= catalog[1].rating >= catalog[2].rating

## 2. Applied AI Case Study: Neural Network Layer Abstraction

Modern deep learning architectures (e.g., PyTorch `nn.Module`) are built on object-oriented composition:
- A base class (`Module`) encapsulates parameters, submodules, and defines the execution protocol (`__call__` -> `forward`).
- Specific layers (`Linear`, `ReLU`) inherit from `Module`, encapsulate weights/biases, and implement their forward transformations.
- Container modules (`Sequential`) compose multiple layers and execute them in sequence.

Let us construct this hierarchy in pure Python using NumPy:

In [ ]:
from abc import ABC, abstractmethod

import numpy as np


class Module(ABC):
    """Abstract base class for all neural network modules."""

    def __init__(self) -> None:
        self._parameters: dict[str, np.ndarray] = {}
        self._submodules: dict[str, Module] = {}

    def register_parameter(self, name: str, param: np.ndarray) -> None:
        self._parameters[name] = param

    @property
    def parameters(self) -> list[np.ndarray]:
        params = list(self._parameters.values())
        for sub in self._submodules.values():
            params.extend(sub.parameters)
        return params

    @abstractmethod
    def forward(self, x: np.ndarray) -> np.ndarray:
        """Subclasses must implement the forward computation."""
        pass

    def __call__(self, x: np.ndarray) -> np.ndarray:
        """Allows module instance to be called directly: output = model(x)."""
        return self.forward(x)


class Linear(Module):
    """Fully connected linear layer: y = x @ W + b."""

    def __init__(self, in_features: int, out_features: int, seed: int = 42) -> None:
        super().__init__()
        rng = np.random.default_rng(seed)
        # Xavier initialization
        limit = np.sqrt(6.0 / (in_features + out_features))
        weights = rng.uniform(-limit, limit, size=(in_features, out_features))
        biases = np.zeros((1, out_features))

        self.register_parameter("weights", weights)
        self.register_parameter("biases", biases)

    def forward(self, x: np.ndarray) -> np.ndarray:
        weights = self._parameters["weights"]
        biases = self._parameters["biases"]
        return np.dot(x, weights) + biases


class ReLU(Module):
    """Rectified Linear Unit activation: max(0, x)."""

    def forward(self, x: np.ndarray) -> np.ndarray:
        return np.maximum(0.0, x)


class Sequential(Module):
    """Composite container executing layers sequentially."""

    def __init__(self, *layers: Module) -> None:
        super().__init__()
        for idx, layer in enumerate(layers):
            self._submodules[f"layer_{idx}"] = layer

    def forward(self, x: np.ndarray) -> np.ndarray:
        out = x
        for layer in self._submodules.values():
            out = layer(out)  # Invokes layer.__call__
        return out


# Test the OOP Neural Network architecture
rng = np.random.default_rng(100)
batch_size, in_dim, hidden_dim, out_dim = 4, 3, 5, 2
dummy_input = rng.standard_normal((batch_size, in_dim))

mlp = Sequential(
    Linear(in_features=in_dim, out_features=hidden_dim, seed=1),
    ReLU(),
    Linear(in_features=hidden_dim, out_features=out_dim, seed=2),
)

output = mlp(dummy_input)

print(f"Input shape:  {dummy_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Total parameter tensors: {len(mlp.parameters)}")
for idx, p in enumerate(mlp.parameters):
    print(f"  Param {idx}: shape {p.shape}")

assert output.shape == (batch_size, out_dim)
assert len(mlp.parameters) == 4  # 2 weights + 2 biases
print("Deep learning OOP hierarchy verified successfully!")

## Summary

- Custom objects can be sorted flexibly via lambda functions, `operator.attrgetter`, or natural dunders (`__lt__`, `@total_ordering`).
- Real-world AI frameworks leverage OOP extensively: encapsulation of learnable parameters, abstract interfaces, and callable instances (`__call__`).
- Combining encapsulation, composition, and dunder protocols yields extensible, modular, production-grade systems.